<p align="center">
  <a href="https://colab.research.google.com/github/NGEO-IDEFLOR-Bio/ngeo_satellite_downloader/blob/main/ngeo_satellite_downloader.ipynb" target="_blank">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Google Colab" width="180">
  </a>
</p>

# NGEO Satellite Downloader

Notebook para o Google Colab destinado à aquisição de imagens de satélite sobre áreas de interesse definidas por shapefile. O fluxo prepara composições, aplica máscaras de nuvens quando disponíveis, divide áreas extensas em tiles e exporta os resultados em GeoTIFF para o Google Drive.

O projeto foi organizado para apoiar rotinas de geotecnologias do **Núcleo de Geotecnologias (NGEO) do IDEFLOR-Bio**, sob coordenação de **Samuel da Costa dos Santos**. Ele deve ser entendido como uma ferramenta operacional: a qualidade do resultado depende da área de estudo, do período escolhido, da disponibilidade das coleções e da validação técnica posterior.

## Sensores e resoluções de trabalho

| Opção | Resolução usada na exportação | Observação |
|---|---:|---|
| Sentinel-2 | 10 m | Permite composição semestral ou análise por meses. |
| Landsat 5/7/8/9 | 30 m | Permite seleção de semestre. |
| CBERS-4A (MUX/WPM) | 2 m | Requer credenciais do portal do INPE durante a sessão. |
| SPOT 2008 — Código Florestal | 5 m | Mosaico específico de 2008; não depende do método de composição. |

## Ordem geral de uso

1. Abra o arquivo no Google Colab e conecte-o a um ambiente de execução.
2. Execute as células na ordem apresentada.
3. Faça upload de um arquivo `.zip` contendo o shapefile e seus arquivos auxiliares, como `.shp`, `.shx`, `.dbf` e, preferencialmente, `.prj`.
4. Configure sensor, anos, período, método de composição, buffer e pasta de saída.
5. Clique em **INICIAR DOWNLOAD** e acompanhe as tarefas de exportação.

> **Atenção:** o notebook usa autenticação do Google Earth Engine e monta o Google Drive. Antes da execução, confirme que a conta possui acesso ao Earth Engine, que o projeto configurado no código está habilitado e que há espaço suficiente no Drive.


## 1. Instalação das dependências

Execute a célula abaixo uma vez por sessão do Colab. Ela instala as bibliotecas usadas para Earth Engine, widgets, leitura de dados vetoriais, processamento raster e acesso ao catálogo CBERS. A instalação pode levar alguns minutos e, em algumas sessões, o Colab pode solicitar a reinicialização do ambiente.

Se o ambiente for reiniciado, execute novamente esta célula antes de prosseguir para a autenticação.


In [ ]:
# CÉLULA 1 - Instalação de dependências
!pip install geemap --quiet
# Resolve requests dependency conflict with google-colab
!pip install requests==2.32.4 --force-reinstall --quiet
!pip install google-colab==1.0.0 --force-reinstall --quiet
# Install remaining dependencies
!pip install cbers4asat rasterio geopandas pycrs geomet geojson scikit-image --quiet

## 2. Autenticação, funções e preparação do ambiente

A célula seguinte importa as bibliotecas, solicita a autenticação no Google Earth Engine, inicializa o projeto configurado e monta o Google Drive em `/content/drive`. Em seguida, define as funções de leitura da região, estimativa de pixels, divisão em tiles, composição, máscara de nuvens e exportação.

A autenticação ocorre no navegador. Não inclua tokens, senhas ou arquivos de credenciais no repositório. A área é convertida para a referência geográfica esperada pelo fluxo quando necessário; ainda assim, mantenha o `.prj` no ZIP para reduzir ambiguidades.

> **Sobre áreas grandes:** o notebook estima a quantidade de pixels na extensão da região. Quando o limite operacional é excedido, cria uma grade de tiles e exporta cada parte separadamente. Confira os limites e a continuidade espacial dos arquivos resultantes antes de utilizá-los em análises.


In [ ]:
# CÉLULA 2 - Imports, autenticação e funções utilitárias
import ee
import geemap
import datetime
import calendar
import zipfile
import os
import math
import shutil
import requests
import time
import logging
import numpy as np
from google.colab import files, drive
import ipywidgets as widgets
from IPython.display import display, clear_output

logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger('satellite_downloader')

# Autenticação Earth Engine
ee.Authenticate()
ee.Initialize(project='sentinel-landsat-download')
print('✅ Earth Engine inicializado!')

# Montar Google Drive
drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/Satellite_Downloads'
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'📁 Pasta Drive: {DRIVE_BASE}')

# --- Variáveis globais ---
SHAPEFILE_REGION = None


# --- Funções utilitárias ---

def parse_years(text):
    years = []
    text = text.strip()
    if '-' in text:
        parts = text.split('-')
        start, end = int(parts[0].strip()), int(parts[1].strip())
        years = list(range(start, end + 1))
    elif ',' in text:
        years = [int(y.strip()) for y in text.split(',')]
    else:
        years = [int(text)]
    return years


def expand_region(region, scale_factor=2.0):
    bounds = region.bounds()
    coords = bounds.coordinates().get(0).getInfo()
    xs = [pt[0] for pt in coords]
    ys = [pt[1] for pt in coords]
    xmin, xmax = min(xs), max(xs)
    ymin, ymax = min(ys), max(ys)
    xmid = (xmin + xmax) / 2
    ymid = (ymin + ymax) / 2
    xrange = (xmax - xmin) * scale_factor / 2
    yrange = (ymax - ymin) * scale_factor / 2
    return ee.Geometry.Rectangle([xmid - xrange, ymid - yrange,
                                  xmid + xrange, ymid + yrange])


# --- Funções de máscara de nuvem e composição inteligente ---

def calc_cloud_in_roi_landsat(image, region, scale=30):
    """Calcula % de nuvens REAIS na ROI usando QA_PIXEL (Landsat)."""
    qa = image.select('QA_PIXEL')
    cloud_mask = qa.bitwiseAnd(1 << 4).neq(0)
    cloud_pct = cloud_mask.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=scale,
        maxPixels=1e6
    ).get('QA_PIXEL')
    return image.set('ROI_CLOUD', cloud_pct)


def apply_cloud_mask_landsat(image):
    """Máscara de nuvem/sombra/neve via QA_PIXEL (Landsat)."""
    qa = image.select('QA_PIXEL')
    mask = qa.bitwiseAnd(1 << 3).eq(0) \
          .And(qa.bitwiseAnd(1 << 4).eq(0)) \
          .And(qa.bitwiseAnd(1 << 5).eq(0))
    return image.updateMask(mask)


def calc_cloud_in_roi_sentinel(image, region, scale=10):
    """Calcula % de nuvens REAIS na ROI usando SCL (Sentinel)."""
    scl = image.select('SCL')
    cloud_mask = scl.eq(7).Or(scl.eq(8)).Or(scl.eq(9)).Or(scl.eq(10))
    cloud_pct = cloud_mask.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=scale,
        maxPixels=1e6
    ).get('SCL')
    return image.set('ROI_CLOUD', cloud_pct)


def apply_cloud_mask_sentinel(image):
    """Máscara de nuvem/sombra via SCL (Sentinel)."""
    scl = image.select('SCL')
    mask = scl.neq(3).And(scl.neq(7)).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return image.updateMask(mask)


def mask_zero_pixels(image):
    """Mascara pixels com valor 0 nas bandas principais (RGB/Refletância)."""
    all_bands = image.bandNames()
    good_bands = all_bands.filter(ee.Filter.stringContains('item', 'QA').Not())\
                          .filter(ee.Filter.stringContains('item', 'SCL').Not())\
                          .filter(ee.Filter.stringContains('item', 'cloud').Not())\
                          .filter(ee.Filter.stringContains('item', 'CLOUD').Not())
    mask = image.select(good_bands).reduce(ee.Reducer.max()).neq(0)
    return image.updateMask(mask)


def smart_composite(collection, region, calc_cloud_fn, mask_fn, scale=30, min_valid_pct=30, max_images=5):
    """Composição inteligente: top5 por nuvens na ROI, mediana das mascaradas."""
    bounds = region.bounds()
    collection_with_roi = collection.map(lambda img: calc_cloud_fn(img, bounds, scale))
    ranked = collection_with_roi.sort('ROI_CLOUD')

    try:
        info_list = ranked.limit(max_images).getInfo().get('features', [])
    except Exception as e:
        print(f'      ⚠️ Erro ao obter metadados das imagens: {e}')
        return None

    n_images = len(info_list)
    if n_images == 0:
        return None

    valid_images = []

    for idx, feat in enumerate(info_list):
        img_id = feat.get('id')
        if not img_id:
            continue
        properties = feat.get('properties', {})
        roi_cloud = properties.get('ROI_CLOUD')
        if roi_cloud is None:
            continue
        valid_pct = (1 - roi_cloud) * 100
        if idx == 0:
            img = ee.Image(img_id)
            valid_images.append(mask_zero_pixels(mask_fn(img)))
        else:
            if valid_pct < min_valid_pct:
                print(f'      img {idx+1}: {valid_pct:.0f}% validos na ROI — descartada')
                continue
            print(f'      img {idx+1}: {valid_pct:.0f}% validos — aceita')
            img = ee.Image(img_id)
            valid_images.append(mask_zero_pixels(mask_fn(img)))

    if not valid_images:
        return None

    combined = ee.ImageCollection(valid_images).median()
    print(f'      ({len(valid_images)} imagens na composição - Otimizado v2)')

    return combined


# --- Exportação para Google Drive (sem limite de tamanho) ---

def export_to_drive(image, description, folder, region, scale=30):
    """Exporta imagem direto do GEE para Google Drive (sem limite de 50MB)."""
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder=folder,
        region=region,
        scale=scale,
        maxPixels=1e13,
        fileFormat='GeoTIFF'
    )
    task.start()
    print(f'    📤 Task criada: {description}')
    return task


def monitor_task(task, description, timeout_min=30):
    """Monitora task do GEE até completar ou timeout."""
    import time as _time
    start = _time.time()
    timeout_sec = timeout_min * 60

    while True:
        status = task.status()
        state = status['state']

        if state == 'COMPLETED':
            print(f'    ✅ Exportação concluída: {description}')
            return True
        elif state in ('FAILED', 'CANCELLED'):
            error_msg = status.get('error_message', 'Erro desconhecido')
            print(f'    ❌ Exportação falhou: {error_msg}')
            return False

        elapsed = (_time.time() - start) / 60
        if elapsed > timeout_min:
            print(f'    ⏰ Timeout ({timeout_min}min). Task ainda rodando em background.')
            return False

        print(f'    ⏳ {state}... ({elapsed:.0f}min)')
        _time.sleep(30)


def get_download_or_export(image, region, scale, output_path, buffer_factor=2.0, drive_folder='Satellite_Downloads'):
    """Exporta para Drive via GEE (sem limite de tamanho)."""
    expanded = expand_region(region, buffer_factor)
    filename = os.path.basename(output_path).replace('.tif', '')
    task = export_to_drive(image, filename, drive_folder, expanded, scale)

    if monitor_task(task, filename):
        print(f'    📁 Arquivo no Drive: {drive_folder}/{filename}')
        return True
    return False


def estimate_pixel_count(region, scale):
    bounds = region.bounds().coordinates().get(0).getInfo()
    xs = [pt[0] for pt in bounds]
    ys = [pt[1] for pt in bounds]
    lat_mid = (min(ys) + max(ys)) / 2
    m_per_deg_lon = 111320 * math.cos(math.radians(lat_mid))
    m_per_deg_lat = 110540
    width_m = (max(xs) - min(xs)) * m_per_deg_lon
    height_m = (max(ys) - min(ys)) * m_per_deg_lat
    return int((width_m / scale) * (height_m / scale))


def split_region_into_tiles(region, scale, max_pixels=5000000):
    total_pixels = estimate_pixel_count(region, scale)
    if total_pixels <= max_pixels:
        return [region], total_pixels
    n_tiles_needed = math.ceil(total_pixels / max_pixels)
    n_side = math.ceil(math.sqrt(n_tiles_needed))
    bounds = region.bounds().coordinates().get(0).getInfo()
    xs = [pt[0] for pt in bounds]
    ys = [pt[1] for pt in bounds]
    xmin, xmax = min(xs), max(xs)
    ymin, ymax = min(ys), max(ys)
    dx = (xmax - xmin) / n_side
    dy = (ymax - ymin) / n_side
    tiles = []
    for i in range(n_side):
        for j in range(n_side):
            tile = ee.Geometry.Rectangle([
                xmin + j * dx, ymin + i * dy,
                xmin + (j + 1) * dx, ymin + (i + 1) * dy
            ])
            tiles.append(tile)
    print(f'  📐 Área grande ({total_pixels:,} px). Dividindo em {len(tiles)} tiles ({n_side}x{n_side}).')
    return tiles, total_pixels


def download_image(url, output_path, timeout=120):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    for attempt in range(3):
        try:
            response = requests.get(url, stream=True, timeout=timeout)
            if response.status_code == 200:
                with open(output_path, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)
                return True
            elif response.status_code == 429:
                print(f'  ⏳ Rate limit. Aguardando 30s...')
                time.sleep(30)
            else:
                print(f'  ❌ HTTP {response.status_code}')
                return False
        except Exception as e:
            print(f'  ⚠️ Tentativa {attempt+1}/3: {e}')
            time.sleep(10)
    return False


def download_tiled_image(image, region, scale, output_path, buffer_factor=2.0):
    expanded = expand_region(region, buffer_factor)
    tiles, total_pixels = split_region_into_tiles(expanded, scale)
    if len(tiles) == 1:
        url = image.clip(expanded).getDownloadURL({
            'scale': scale, 'region': expanded, 'format': 'GeoTIFF'
        })
        return download_image(url, output_path)
    tile_paths = []
    for idx, tile in enumerate(tiles):
        tile_path = output_path.replace('.tif', f'_tile{idx:03d}.tif')
        print(f'  📥 Tile {idx+1}/{len(tiles)}...')
        try:
            url = image.clip(tile).getDownloadURL({
                'scale': scale, 'region': tile, 'format': 'GeoTIFF'
            })
            if download_image(url, tile_path):
                tile_paths.append(tile_path)
        except Exception as e:
            print(f'  ❌ Erro tile {idx+1}: {e}')
    if not tile_paths:
        return False
    print(f'  🔗 Mergeando {len(tile_paths)} tiles...')
    try:
        import rasterio
        from rasterio.merge import merge
        src_files = [rasterio.open(p) for p in tile_paths]
        mosaic, out_transform = merge(src_files)
        out_profile = src_files[0].profile.copy()
        out_profile.update({'driver': 'GTiff', 'height': mosaic.shape[1],
                           'width': mosaic.shape[2], 'transform': out_transform})
        with rasterio.open(output_path, 'w', **out_profile) as dest:
            dest.write(mosaic)
        for src in src_files:
            src.close()
        for p in tile_paths:
            if os.path.exists(p): os.remove(p)
        print(f'  ✅ Merge concluído')
        return True
    except Exception as e:
        print(f'  ❌ Erro merge: {e}')
        return True


def get_sentinel_image(region, year, start_month, end_month, method='median'):
    """Obtém imagem Sentinel-2 SR com composição inteligente."""
    _, last_day = calendar.monthrange(year, end_month)
    start_date = f'{year}-{start_month:02d}-01'
    end_date = f'{year}-{end_month:02d}-{last_day}'

    # NÃO seleciona bandas — mantém QA_PIXEL para a máscara
    bounds = region.bounds()
    collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                  .filterBounds(bounds)
                  .filterDate(start_date, end_date)
                  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80)))

    count = collection.size().getInfo()
    if count == 0:
        return None

    print(f'    ({count} imagens encontradas)')

    if method == 'best':
        return collection.sort('CLOUDY_PIXEL_PERCENTAGE').first().select(['B4', 'B3', 'B2'])

    print(f'    Composição inteligente (top5, nuvens na ROI):')
    img = smart_composite(collection, region, calc_cloud_in_roi_sentinel, apply_cloud_mask_sentinel, scale=10, min_valid_pct=30, max_images=5)
    return img.select(['B4', 'B3', 'B2']) if img else None


def get_landsat_image(region, year, semester, method='median'):
    """Obtém imagem Landsat (igual ao plugin original)."""
    if semester == 1:
        start_date = f'{year}-01-01'
        end_date = f'{year}-06-30'
    else:
        start_date = f'{year}-07-01'
        end_date = f'{year}-12-31'

    if year <= 2011:
        collection_id = 'LANDSAT/LT05/C02/T1_L2'
        band_options = ['SR_B5', 'SR_B4', 'SR_B3']
    elif year <= 2013:
        collection_id = 'LANDSAT/LE07/C02/T1_L2'
        band_options = ['SR_B3', 'SR_B2', 'SR_B1']
    elif year <= 2021:
        collection_id = 'LANDSAT/LC08/C02/T1_L2'
        band_options = ['SR_B6', 'SR_B5', 'SR_B4']
    else:
        collection_id = 'LANDSAT/LC09/C02/T1_L2'
        band_options = ['SR_B4', 'SR_B3', 'SR_B2']

    bounds = region.bounds()
    collection = (ee.ImageCollection(collection_id)
                  .filterBounds(bounds)
                  .filterDate(start_date, end_date)
                  .filter(ee.Filter.lt('CLOUD_COVER', 10)))

    count = collection.size().getInfo()
    if count == 0:
        return None, None

    if method == 'best':
        image = collection.sort('CLOUD_COVER').first()
        return image.select(band_options), band_options

    image = collection.median().select(band_options)
    return image, band_options


def get_spot_2008_image(region, buffer_factor=2.0):
    """Obtém mosaico SPOT 2008 (igual ao plugin)."""
    try:
        image = ee.Image('GOOGLE/BRAZIL_FOREST_2008/V1/VISUAL')
        expanded = expand_region(region, buffer_factor)
        clipped = image.clip(expanded)
        clipped.getInfo()
        return clipped.select(['R', 'G', 'B'])
    except Exception as e:
        print(f'  ❌ Erro SPOT 2008: {e}')
        return None


def check_cbers_deps():
    try:
        import cbers4asat
        import rasterio
        import geopandas
        import shapely
        return True, ''
    except ImportError as e:
        return False, str(e)


def get_cbers_image_inpe(region_ee, year, months, output_dir, scale_factor=2, inpe_email=None):
    has_deps, err = check_cbers_deps()
    if not has_deps:
        print(f'  ❌ Dependências CBERS: {err}')
        return None
    try:
        from cbers4asat import Cbers4aAPI
        from cbers4asat.tools import rgbn_composite, clip as raster_clip
        from shapely.geometry import Polygon
        import geopandas as gpd
        import rasterio as rio

        if not inpe_email:
            print('  ❌ Email do INPE não configurado.')
            return None

        bounds = region_ee.bounds().coordinates().get(0).getInfo()
        lons = [p[0] for p in bounds]
        lats = [p[1] for p in bounds]
        xmid = (min(lons) + max(lons)) / 2
        ymid = (min(lats) + max(lats)) / 2
        xrange = (max(lons) - min(lons)) * scale_factor / 2
        yrange = (max(lats) - min(lats)) * scale_factor / 2
        bbox = [xmid - xrange, ymid - yrange, xmid + xrange, ymid + yrange]
        poly_aoi = Polygon([(bbox[0], bbox[1]), (bbox[2], bbox[1]),
                            (bbox[2], bbox[3]), (bbox[0], bbox[3])])

        api = Cbers4aAPI(inpe_email)
        start_date = datetime.date(year, min(months), 1)
        _, last_day = calendar.monthrange(year, max(months))
        end_date = datetime.date(year, max(months), last_day)
        print(f'  🔍 Buscando CBERS-4A ({start_date} a {end_date})...')

        products = api.query(location=bbox, initial_date=start_date,
                             end_date=end_date, cloud=100, limit=50,
                             collections=['CBERS4A_WPM_L4_DN'])
        if not products or not products.get('features'):
            print('  ⚠️ Nenhuma imagem CBERS encontrada.')
            return None

        features = products['features']
        def get_cloud(f):
            p = f.get('properties', {})
            return p.get('cloud_cover', p.get('eo:cloud_cover', 100))
        features.sort(key=get_cloud)
        best = features[0]
        scene_id = best['id']
        cloud_val = get_cloud(best)
        date_str = best['properties'].get('datetime', '').split('T')[0]
        print(f'  ✨ Melhor cena: {scene_id} ({date_str}, {cloud_val}% nuvens)')

        bands = ['red', 'green', 'blue', 'nir']
        temp_dir = os.path.join(output_dir, 'temp_cbers')
        os.makedirs(temp_dir, exist_ok=True)
        print(f'  📥 Baixando bandas...')
        api.download(products={'type': 'FeatureCollection', 'features': [best]},
                     bands=bands, outdir=temp_dir, with_folder=True)

        scene_subdir = os.path.join(temp_dir, scene_id)
        if not os.path.exists(scene_subdir):
            subdirs = [d for d in os.listdir(temp_dir) if os.path.isdir(os.path.join(temp_dir, d))]
            if subdirs: scene_subdir = os.path.join(temp_dir, subdirs[0])
        files_list = os.listdir(scene_subdir)
        b_red = next((os.path.join(scene_subdir, f) for f in files_list if 'BAND3' in f or 'BAND5' in f), None)
        b_green = next((os.path.join(scene_subdir, f) for f in files_list if 'BAND2' in f or 'BAND6' in f), None)
        b_blue = next((os.path.join(scene_subdir, f) for f in files_list if 'BAND1' in f or 'BAND7' in f), None)
        b_nir = next((os.path.join(scene_subdir, f) for f in files_list if 'BAND4' in f or 'BAND8' in f), None)
        if not all([b_red, b_green, b_blue]):
            print('  ❌ Erro ao localizar bandas.')
            return None

        composite_path = os.path.join(output_dir, f'CBERS_{scene_id}_STACK.tif')
        print(f'  🎨 Criando composição RGB...')
        rgbn_composite(red=b_red, green=b_green, blue=b_blue, nir=b_nir,
                       filename=os.path.basename(composite_path), outdir=output_dir)

        final_filename = f'CBERS_{year}_{scene_id[:10]}.tif'
        final_path = os.path.join(output_dir, final_filename)
        print(f'  ✂️ Recortando...')
        try:
            with rio.open(composite_path) as src:
                raster_crs = src.crs or 'EPSG:3857'
            gdf_aoi = gpd.GeoDataFrame(index=[0], crs='EPSG:4326', geometry=[poly_aoi])
            gdf_aoi = gdf_aoi.to_crs(raster_crs)
            poly_mask = gdf_aoi.geometry.iloc[0]
            raster_clip(raster=composite_path, mask=poly_mask,
                       filename=final_filename, outdir=output_dir)
        except Exception as e:
            print(f'  ⚠️ Erro recorte: {e}. Fallback...')
            raster_clip(raster=composite_path, mask=poly_aoi,
                       filename=final_filename, outdir=output_dir)

        try:
            shutil.rmtree(temp_dir)
            if os.path.exists(composite_path): os.remove(composite_path)
        except: pass
        return final_path
    except Exception as e:
        print(f'  ❌ Erro CBERS: {e}')


## 3. Interface interativa e parâmetros

Depois de executar a célula abaixo, a interface aparece no final do notebook. Use primeiro o botão de upload e aguarde a confirmação dos limites da região. Só então selecione os demais parâmetros.

- **Satélite:** Sentinel, Landsat, CBERS-4A ou SPOT 2008.
- **Anos:** um ano (`2024`), uma faixa (`2020-2024`) ou uma lista separada por vírgulas (`2020, 2022, 2024`).
- **Modo:** composição por semestre ou análise anual por meses. O modo mensal fica disponível para Sentinel e CBERS.
- **Semestre:** primeiro, segundo ou ambos, quando o sensor selecionado oferecer essa opção.
- **Método:** mediana, geralmente mais robusta a ruídos residuais, ou melhor imagem, selecionando a menor cobertura de nuvens disponível.
- **Buffer:** fator de expansão da extensão exportada. O valor padrão é `2.0`; aumente-o somente quando houver uma justificativa espacial clara.
- **Pasta Drive:** nome da pasta no Google Drive onde as tarefas do Earth Engine depositarão os GeoTIFFs.

Para CBERS-4A, informe as credenciais do portal do INPE apenas no widget exibido para esse sensor. A senha é usada na sessão e não deve ser salva em células, logs ou arquivos. O e-mail não vem mais pré-preenchido nesta versão.


In [ ]:
# CÉLULA 3 - Interface interativa

def create_ui():
    global SHAPEFILE_REGION

    upload_output = widgets.Output()

    def upload_shapefile(b):
        global SHAPEFILE_REGION
        with upload_output:
            clear_output()
            print('Faça upload de um .zip com o shapefile...')
            if os.path.exists('shapefile'):
                shutil.rmtree('shapefile')
            os.makedirs('shapefile')
            uploaded = files.upload()
            for filename in uploaded.keys():
                if filename.endswith('.zip'):
                    with zipfile.ZipFile(filename, 'r') as zip_ref:
                        zip_ref.extractall('shapefile')
                    print('✅ Shapefile extraído!')
                    break
            else:
                print('❌ Nenhum .zip encontrado.')
                return
            shp_files = [f for f in os.listdir('shapefile') if f.endswith('.shp')]
            if not shp_files:
                print('❌ .shp não encontrado no zip.')
                return
            shp_path = os.path.join('shapefile', shp_files[0])
            feature = None
            try:
                feature = geemap.shp_to_ee(shp_path)
            except Exception as e:
                print(f'  ⚠️ geemap.shp_to_ee falhou: {e}')
            if feature is None:
                print('  ℹ️ Tentando carregar shapefile com geopandas como fallback...')
                try:
                    import geopandas as gpd
                    import json as _json
                    gdf = gpd.read_file(shp_path)
                    if gdf.crs is None:
                        gdf = gdf.set_crs('EPSG:4326')
                    elif gdf.crs.to_string() != 'EPSG:4326':
                        gdf = gdf.to_crs('EPSG:4326')
                    feature = ee.FeatureCollection(_json.loads(gdf.to_json()))
                except Exception as gpd_err:
                    print(f'  ❌ Erro ao carregar com geopandas: {gpd_err}')
                    return
            SHAPEFILE_REGION = feature.geometry()
            bounds = SHAPEFILE_REGION.bounds().coordinates().get(0).getInfo()
            xs = [pt[0] for pt in bounds]
            ys = [pt[1] for pt in bounds]
            print(f'📍 Bounds: X[{min(xs):.4f}, {max(xs):.4f}] Y[{min(ys):.4f}, {max(ys):.4f}]')
            for name, scale in [('Sentinel (10m)', 10), ('Landsat (30m)', 30), ('SPOT (5m)', 5)]:
                px = estimate_pixel_count(SHAPEFILE_REGION, scale)
                status = '✅ OK' if px < 5000000 else '⚠️ Será dividido em tiles'
                print(f'  {name}: {px:,} px {status}')

    upload_btn = widgets.Button(description='📁 Upload Shapefile (.zip)',
                                button_style='info',
                                layout=widgets.Layout(width='300px', height='40px'))
    upload_btn.on_click(upload_shapefile)

    sat_dropdown = widgets.Dropdown(
        options=['Sentinel', 'Landsat', 'CBERS-4A (MUX/WPM)', 'SPOT 2008 (Código Florestal)'],
        value='Sentinel', description='🛰️ Satélite:',
        style={'description_width': '120px'}, layout=widgets.Layout(width='400px'))

    years_text = widgets.Text(value='2024', description='📅 Anos:',
                              placeholder='Ex: 2024 ou 2020-2024',
                              style={'description_width': '120px'},
                              layout=widgets.Layout(width='400px'))

    mode_radio = widgets.RadioButtons(
        options=['Composição por Semestre', 'Análise Anual (Meses)'],
        value='Composição por Semestre', description='📋 Modo:',
        style={'description_width': '120px'}, layout=widgets.Layout(width='400px'))

    semester_dropdown = widgets.Dropdown(
        options=['1º Semestre', '2º Semestre', 'Ambos'],
        value='Ambos', description='📆 Semestre:',
        style={'description_width': '120px'}, layout=widgets.Layout(width='400px'))

    months = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun',
              'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
    month_checks = []
    for i, name in enumerate(months):
        cb = widgets.Checkbox(value=(i in [5, 6, 7]), description=name,
                              layout=widgets.Layout(width='100px'))
        month_checks.append(cb)
    month_grid = widgets.GridBox(
        children=month_checks,
        layout=widgets.Layout(grid_template_columns='repeat(4, 100px)',
                              grid_gap='5px', width='420px'))
    month_container = widgets.VBox([
        widgets.HTML(value='<b>Selecione os meses:</b>'), month_grid])
    month_container.layout.display = 'none'

    method_dropdown = widgets.Dropdown(
        options=[('Mediana (Mais limpa)', 'median'),
                 ('Melhor Imagem (Menor Nuvens)', 'best')],
        value='median', description='🔬 Método:',
        style={'description_width': '120px'}, layout=widgets.Layout(width='400px'))

    buffer_spin = widgets.FloatSlider(
        value=2.0, min=1.0, max=10.0, step=0.5,
        description='🔲 Buffer:', style={'description_width': '120px'},
        layout=widgets.Layout(width='400px'))

    folder_text = widgets.Text(
        value='Satellite_Downloads', description='📁 Pasta Drive:',
        placeholder='Nome da pasta no Google Drive',
        style={'description_width': '120px'}, layout=widgets.Layout(width='400px'))

    # --- Credenciais INPE (apenas para CBERS) ---
    inpe_email_text = widgets.Text(
        value='',
        description='📧 Email INPE:',
        placeholder='seu@email.com',
        style={'description_width': '120px'},
        layout=widgets.Layout(width='400px'))

    inpe_pass_text = widgets.Password(
        value='',
        description='🔑 Senha INPE:',
        placeholder='Digite sua senha do portal INPE',
        style={'description_width': '120px'},
        layout=widgets.Layout(width='400px'))

    inpe_creds_container = widgets.VBox([
        widgets.HTML(value='<b>Credenciais INPE (para CBERS-4A):</b>'),
        inpe_email_text,
        inpe_pass_text,
        widgets.HTML(value='<small style="color:#888">A senha é usada apenas nesta sessão e não é salva.</small>')
    ])
    inpe_creds_container.layout.display = 'none'  # Escondido por padrão

    download_btn = widgets.Button(
        description='🚀 INICIAR DOWNLOAD', button_style='success',
        layout=widgets.Layout(width='300px', height='50px'))

    log_output = widgets.Output(
        layout=widgets.Layout(height='300px', overflow_y='auto',
                              border='1px solid #ccc', padding='10px'))

    progress = widgets.IntProgress(
        value=0, min=0, max=100, description='Progresso:',
        bar_style='success', style={'description_width': '120px'},
        layout=widgets.Layout(width='400px'))

    def on_satellite_change(change):
        sat = change['new']
        is_landsat = 'Landsat' in sat
        is_spot = 'SPOT' in sat
        is_cbers = 'CBERS' in sat
        semester_dropdown.layout.display = '' if is_landsat else 'none'
        mode_radio.layout.display = 'none' if (is_landsat or is_spot) else ''
        if is_landsat or is_spot:
            month_container.layout.display = 'none'
        elif mode_radio.value == 'Análise Anual (Meses)':
            month_container.layout.display = ''
        method_dropdown.disabled = is_cbers or is_spot
        # Mostrar/esconder credenciais INPE
        inpe_creds_container.layout.display = '' if is_cbers else 'none'

    def on_mode_change(change):
        is_monthly = change['new'] == 'Análise Anual (Meses)'
        sat = sat_dropdown.value
        if 'Landsat' in sat or 'SPOT' in sat:
            month_container.layout.display = 'none'
        else:
            month_container.layout.display = '' if is_monthly else 'none'

    sat_dropdown.observe(on_satellite_change, names='value')
    mode_radio.observe(on_mode_change, names='value')
    on_satellite_change({'new': sat_dropdown.value})

    def run_download(b):
        download_btn.disabled = True
        progress.value = 0
        progress.bar_style = 'info'

        with log_output:
            clear_output()
            print('=' * 60)
            print('🛰️ QGIS SATELLITE DOWNLOADER - COLAB (VERSÃO OTIMIZADA V2)')
            print('=' * 60)

            if SHAPEFILE_REGION is None:
                print('❌ ERRO: Faça upload do shapefile primeiro!')
                download_btn.disabled = False
                return

            region = SHAPEFILE_REGION
            sat = sat_dropdown.value
            years = parse_years(years_text.value)
            buffer_factor = buffer_spin.value
            method = method_dropdown.value
            folder_name = folder_text.value.strip() or 'Satellite_Downloads'

            is_sentinel = 'Sentinel' in sat
            is_landsat = 'Landsat' in sat
            is_cbers = 'CBERS' in sat
            is_spot = 'SPOT' in sat

            # Validar credenciais INPE se CBERS selecionado
            inpe_email = None
            if is_cbers:
                inpe_email = inpe_email_text.value.strip()
                if not inpe_email:
                    print('❌ ERRO: Preencha o email do INPE para CBERS!')
                    download_btn.disabled = False
                    return
                inpe_pass = inpe_pass_text.value
                if not inpe_pass:
                    print('⚠️ AVISO: Senha do INPE não preenchida. Tentando sem senha...')
                else:
                    print(f'📧 Email INPE: {inpe_email}')
                    print('🔑 Senha INPE: [configurada]')

            if is_sentinel: scale = 10
            elif is_landsat: scale = 30
            elif is_spot: scale = 5
            else: scale = 2

            output_dir = os.path.join('/content/drive/MyDrive', folder_name)
            os.makedirs(output_dir, exist_ok=True)

            bounds = region.bounds().coordinates().get(0).getInfo()
            xs = [pt[0] for pt in bounds]
            ys = [pt[1] for pt in bounds]
            area_label = f'Area_{min(xs):.2f}_{min(ys):.2f}_{max(xs):.2f}_{max(ys):.2f}'
            area_dir = os.path.join(output_dir, area_label)
            os.makedirs(area_dir, exist_ok=True)

            print(f'\n📋 Configurações:')
            print(f'   Satélite: {sat}')
            print(f'   Anos: {years}')
            print(f'   Escala: {scale}m | Buffer: {buffer_factor}x | Método: {method}')
            print(f'   Pasta: {area_dir}')

            total_ops = 0
            for year in years:
                if is_sentinel or is_cbers:
                    if mode_radio.value == 'Composição por Semestre':
                        total_ops += 2
                    else:
                        total_ops += sum(1 for cb in month_checks if cb.value)
                elif is_landsat:
                    sem = semester_dropdown.value
                    total_ops += 1 if sem != 'Ambos' else 2
                elif is_spot:
                    total_ops += 1

            current_op = 0
            print(f'\n🔄 Total de operações: {total_ops}')
            print('-' * 60)

            for year in years:
                print(f'\n📅 === ANO {year} ===')

                if is_sentinel:
                    if mode_radio.value == 'Composição por Semestre':
                        for sem in [1, 2]:
                            current_op += 1
                            start_m, end_m = (1, 6) if sem == 1 else (7, 12)
                            print(f'\n  📥 Sentinel {year} S{sem}...')
                            img = get_sentinel_image(region, year, start_m, end_m, method=method)
                            if img:
                                filename = f'Sentinel_{year}_S{sem}.tif'
                                filepath = os.path.join(area_dir, filename)
                                try:
                                    success = get_download_or_export(img, region, scale, filepath, buffer_factor, folder_name)
                                    if success: print(f'  ✅ Salvo: {filename}')
                                    else: print(f'  ❌ Falha')
                                except Exception as e:
                                    import traceback, sys
                                    print('❌ ERRO DETECTADO - TRACEBACK COMPLETO:', file=sys.stderr)
                                    traceback.print_exc(file=sys.stderr)
                                    raise e
                            else:
                                print(f'  ⚠️ Sem imagem')
                            progress.value = int(current_op / total_ops * 100)
                    else:
                        selected_months = [i+1 for i, cb in enumerate(month_checks) if cb.value]
                        for month in selected_months:
                            current_op += 1
                            print(f'\n  📥 Sentinel {year}-{month:02d}...')
                            img = get_sentinel_image(region, year, month, month, method=method)
                            if img:
                                filename = f'Sentinel_{year}_{month:02d}.tif'
                                filepath = os.path.join(area_dir, filename)
                                try:
                                    success = get_download_or_export(img, region, scale, filepath, buffer_factor, folder_name)
                                    if success: print(f'  ✅ Salvo: {filename}')
                                    else: print(f'  ❌ Falha')
                                except Exception as e:
                                    import traceback, sys
                                    print('❌ ERRO DETECTADO - TRACEBACK COMPLETO:', file=sys.stderr)
                                    traceback.print_exc(file=sys.stderr)
                                    raise e
                            else:
                                print(f'  ⚠️ Sem imagem')
                            progress.value = int(current_op / total_ops * 100)

                elif is_landsat:
                    sem_choice = semester_dropdown.value
                    semesters = [1] if sem_choice == '1º Semestre' else [2] if sem_choice == '2º Semestre' else [1, 2]
                    for sem in semesters:
                        current_op += 1
                        print(f'\n  📥 Landsat {year} S{sem}...')
                        img, bands = get_landsat_image(region, year, sem, method=method)
                        if img:
                            filename = f'Landsat_{year}_S{sem}.tif'
                            filepath = os.path.join(area_dir, filename)
                            try:
                                success = get_download_or_export(img, region, scale, filepath, buffer_factor, folder_name)
                                if success: print(f'  ✅ Salvo: {filename}')
                                else: print(f'  ❌ Falha')
                            except Exception as e:
                                import traceback, sys
                                print('❌ ERRO DETECTADO - TRACEBACK COMPLETO:', file=sys.stderr)
                                traceback.print_exc(file=sys.stderr)
                                raise e
                        else:
                            print(f'  ⚠️ Sem imagem')
                        progress.value = int(current_op / total_ops * 100)

                elif is_cbers:
                    if mode_radio.value == 'Composição por Semestre':
                        for sem in [1, 2]:
                            current_op += 1
                            months = [1,2,3,4,5,6] if sem == 1 else [7,8,9,10,11,12]
                            print(f'\n  📥 CBERS {year} S{sem}...')
                            result = get_cbers_image_inpe(region, year, months, area_dir,
                                                        scale_factor=buffer_factor,
                                                        inpe_email=inpe_email)
                            if result: print(f'  ✅ Salvo: {os.path.basename(result)}')
                            else: print(f'  ⚠️ Sem imagem')
                            progress.value = int(current_op / total_ops * 100)
                    else:
                        selected_months = [i+1 for i, cb in enumerate(month_checks) if cb.value]
                        if selected_months:
                            current_op += 1
                            print(f'\n  📥 CBERS {year}...')
                            result = get_cbers_image_inpe(region, year, selected_months, area_dir,
                                                        scale_factor=buffer_factor,
                                                        inpe_email=inpe_email)
                            if result: print(f'  ✅ Salvo: {os.path.basename(result)}')
                            else: print(f'  ⚠️ Sem imagem')
                            progress.value = int(current_op / total_ops * 100)

                elif is_spot:
                    current_op += 1
                    print(f'\n  📥 SPOT 2008 - Mosaico Código Florestal...')
                    img = get_spot_2008_image(region, buffer_factor)
                    if img:
                        filename = 'SPOT2008_2008_CF.tif'
                        filepath = os.path.join(area_dir, filename)
                        try:
                            success = get_download_or_export(img, region, scale, filepath, buffer_factor, folder_name)
                            if success: print(f'  ✅ Salvo: {filename}')
                            else: print(f'  ❌ Falha')
                        except Exception as e:
                            import traceback, sys
                            print('❌ ERRO DETECTADO - TRACEBACK COMPLETO:', file=sys.stderr)
                            traceback.print_exc(file=sys.stderr)
                            raise e
                    else:
                        print(f'  ⚠️ Sem imagem')
                    progress.value = int(current_op / total_ops * 100)

            progress.value = 100
            progress.bar_style = 'success'
            print(f'\n{"=" * 60}')
            print('✨ PROCESSO CONCLUÍDO!')
            print(f'📁 Arquivos em: {area_dir}')
            print('=' * 60)

            tifs = [f for f in os.listdir(area_dir) if f.endswith('.tif')]
            if tifs:
                print(f'\n📋 Arquivos ({len(tifs)}):')
                for tif in sorted(tifs):
                    size = os.path.getsize(os.path.join(area_dir, tif)) / (1024*1024)
                    print(f'   📄 {tif} ({size:.1f} MB)')

            download_btn.disabled = False

    download_btn.on_click(run_download)

    ui = widgets.VBox([
        widgets.HTML(value='<h1>🛰️ QGIS Satellite Downloader - Colab</h1>'),
        widgets.HTML(value='<p>Baixe imagens de satélite para áreas grandes com divisão automática em tiles.</p>'),
        widgets.HTML(value='<h3>1️⃣ Shapefile de Entrada</h3>'),
        upload_btn, upload_output,
        widgets.HTML(value='<hr>'),
        widgets.HTML(value='<h3>2️⃣ Parâmetros do Satélite</h3>'),
        sat_dropdown, years_text, mode_radio, semester_dropdown,
        month_container, method_dropdown, buffer_spin,
        widgets.HTML(value='<hr>'),
        widgets.HTML(value='<h3>3️⃣ Credenciais INPE (CBERS)</h3>'),
        inpe_creds_container,
        widgets.HTML(value='<hr>'),
        widgets.HTML(value='<h3>4️⃣ Configurações de Saída</h3>'),
        folder_text,
        widgets.HTML(value='<hr>'),
        widgets.HTML(value='<h3>5️⃣ Executar</h3>'),
        download_btn, progress, log_output
    ])
    display(ui)

create_ui()

## 4. Saída e controle de qualidade

Ao iniciar o download, o notebook cria tarefas no Google Earth Engine e acompanha o estado de cada exportação. Os arquivos são GeoTIFFs organizados na pasta escolhida do Drive. Para áreas extensas, espere por múltiplos arquivos correspondentes aos tiles.

Após a conclusão, verifique: (a) se todos os tiles esperados foram gerados; (b) se a nomenclatura identifica sensor, ano e período; (c) se a extensão cobre a área solicitada; (d) se há lacunas provocadas por nuvens, ausência de aquisição ou máscara; e (e) se a resolução e o sistema de referência são adequados ao uso pretendido. O download não substitui a validação cartográfica e temática.

## Créditos

Desenvolvimento e coordenação: **Samuel da Costa dos Santos** — [samuelsantos.site](https://samuelsantos.site/). **Núcleo de Geotecnologias (NGEO) do Instituto de Desenvolvimento Florestal e da Biodiversidade do Estado do Pará (IDEFLOR-Bio)**.
